In [0]:
%run "../Includes/configuration"

In [0]:
%run "../Includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%fs
ls abfss://silver@moviehistory310785.dfs.core.windows.net/

path,name,size,modificationTime
abfss://silver@moviehistory310785.dfs.core.windows.net/__unitystorage/,__unitystorage/,0,0


In [0]:
movie_df = spark.read.table("movie_silver.movies") \
                     .filter(f"file_date = '{v_file_date}'")

In [0]:
production_country_df = spark.read.table("movie_silver.production_country") \
                                   .filter(f"file_date = '{v_file_date}'")

In [0]:
country_df = spark.read.table("movie_silver.countries")

In [0]:
name_production_company_df = production_country_df \
                             .join(country_df,
                                    production_country_df.country_Id == country_df.country_Id,
                                    "inner") \
                             .select(production_country_df.movie_Id,
                                     country_df.country_Name)

In [0]:
movie_name_production_company_df = name_production_company_df \
                                   .join(movie_df,
                                         name_production_company_df.movie_Id == movie_df.movie_Id,
                                         "inner") \
                                   .select(movie_df.budget,
                                           movie_df.revenue,
                                           movie_df.year_Release_Date,
                                           name_production_company_df.country_Name)

In [0]:
display(movie_name_production_company_df)

budget,revenue,year_Release_Date,country_Name
2.1E7,1.001105E7,2002,United States of America
2000000.0,2600000.0,2010,United States of America
2000000.0,2600000.0,2001,United States of America
2000000.0,2600000.0,2004,Germany
2000000.0,2600000.0,2004,United Kingdom
2000000.0,2600000.0,2004,United States of America
2000000.0,2600000.0,2005,United Kingdom
1.8E7,2.25E7,2009,France
1.8E7,2.25E7,2009,Hong Kong
1.8E7,2.25E7,2009,Ireland


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum, rank, dense_rank, desc, lit

In [0]:
movie_name_country_rank = Window.partitionBy("year_Release_Date").orderBy(desc("total_del_Presupuesto"), desc("total_de_Ingresos"))
movie_name_country_dense_rank = Window.partitionBy("year_Release_Date").orderBy(desc("total_del_Presupuesto"), desc("total_de_Ingresos"))

results_group_movie_country = movie_name_production_company_df \
                              .filter("year_Release_Date >= 2015") \
                              .groupBy("year_Release_Date", "country_Name") \
                                  .agg(sum("budget").alias("total_del_Presupuesto"),
                                       sum("revenue").alias("total_de_Ingresos")) \
                              .withColumn("rank", 
                                          rank().over(movie_name_country_rank)
                                          ) \
                              .withColumn("dense_rank", 
                                          dense_rank().over(movie_name_country_dense_rank)
                                          ) \
                              .withColumn("created_date", lit(v_file_date))

In [0]:
display(results_group_movie_country)

year_Release_Date,country_Name,total_del_Presupuesto,total_de_Ingresos,rank,dense_rank,created_date
2015,United States of America,5000000.0,6250000.0,1,1,2024-12-23
2016,United States of America,3.37E8,6.35203772E8,1,1,2024-12-23


In [0]:
# overwrite_partition("movie_gold", "results_group_movie_country", "created_date", v_file_date)

In [0]:
merge_delta_lake_2(results_group_movie_country, "movie_gold", "results_group_movie_country", "year_Release_Date", "country_Name", "created_date")

In [0]:
# results_group_movie_country.write.mode("append").partitionBy("created_date").format("delta").saveAsTable("movie_gold.results_group_movie_country")

In [0]:
display(spark.read.table("movie_gold.results_group_movie_country"))

year_Release_Date,country_Name,total_del_Presupuesto,total_de_Ingresos,rank,dense_rank,created_date
2015,United States of America,5.143775E9,1.8450384353E10,1,1,2024-12-30
2015,United Kingdom,6.5152236E8,1.894996027E9,2,2,2024-12-30
2015,Canada,3.245E8,1.334394558E9,3,3,2024-12-30
2015,Germany,3.095E8,9.5835032E8,4,4,2024-12-30
2015,China,2.8E8,8.8867852E8,5,5,2024-12-30
2015,Australia,2.13E8,6.79034882E8,6,6,2024-12-30
2015,Japan,1.9E8,1.50624936E9,7,7,2024-12-30
2015,France,1.51E8,2.06495048E8,8,8,2024-12-30
2015,Hong Kong,1.35E8,5.32950503E8,9,9,2024-12-30
2015,Taiwan,1.35E8,5.32950503E8,9,9,2024-12-30


In [0]:
%sql
SELECT created_date, COUNT(1)
FROM movie_gold.results_group_movie_country
GROUP BY created_date

created_date,count(1)
2024-12-23,2
2024-12-30,45


In [0]:
%sql
SELECT * FROM movie_gold.results_group_movie_country;

year_Release_Date,country_Name,total_del_Presupuesto,total_de_Ingresos,rank,dense_rank,created_date
2015,United States of America,5.143775E9,1.8450384353E10,1,1,2024-12-30
2015,United Kingdom,6.5152236E8,1.894996027E9,2,2,2024-12-30
2015,Canada,3.245E8,1.334394558E9,3,3,2024-12-30
2015,Germany,3.095E8,9.5835032E8,4,4,2024-12-30
2015,China,2.8E8,8.8867852E8,5,5,2024-12-30
2015,Australia,2.13E8,6.79034882E8,6,6,2024-12-30
2015,Japan,1.9E8,1.50624936E9,7,7,2024-12-30
2015,France,1.51E8,2.06495048E8,8,8,2024-12-30
2015,Hong Kong,1.35E8,5.32950503E8,9,9,2024-12-30
2015,Taiwan,1.35E8,5.32950503E8,9,9,2024-12-30
